# Case 7: Cone dystrophy

*Hands-On Electroretinography in the Age of AI*, Chapter 19, §19.7.

Mr. A. is in the ≤35y age stratum. His photopic (cone) responses are small, so this notebook first checks the signal quality of each protocol with the pipeline's silver thread SNR thresholds. It then assigns the zones of Table 19.8, checks zones and printed limits against the book, and shows the diagnostic signature: RED photopic responses beside GREEN scotopic responses.

The values are constructed for teaching; they are not patient data. Limits: Baker et al. (2025), silver thread, ≤35y.

**Setup.** In Google Colab, the first cell clones the repository. Elsewhere, run the notebook from the repository.

In [1]:
import os, sys
if 'google.colab' in sys.modules and not os.path.exists('/content/ERG-Analysis-API'):
    !git clone -q https://github.com/AI-Fairness-com/ERG-Analysis-API.git /content/ERG-Analysis-API
sys.path.insert(0, '/content/ERG-Analysis-API/examples')
sys.path.insert(0, os.path.abspath('examples') if os.path.isdir('examples') else os.getcwd())
from case_helpers import zone_table, check, REPORT, erg

**Signal quality (§19.7.1).** For silver thread (DTL), an SNR of 4 or more passes (Chapter 2). The photopic values are lower because the cone responses are small, but all three pass.

In [2]:
for protocol, snr in [('DA 3.0', 14.2), ('LA 3.0', 5.4), ('LA 30 Hz', 4.8)]:
    status, note = erg.CONFIG.get_electrode_snr_status('dtl_fiber', snr)
    print(f'{protocol:9s} {status:5s} {note}')

DA 3.0    PASS  SNR 14.2 ≥ 4.0 threshold
LA 3.0    PASS  SNR 5.4 ≥ 4.0 threshold
LA 30 Hz  PASS  SNR 4.8 ≥ 4.0 threshold


**Table 19.8 values.**

In [3]:
ROWS = [
    ('DA 0.01', 'b-wave amplitude',      215,  'GREEN', (138, 297)),
    ('DA 0.01', 'b-wave implicit time',  84,   'GREEN', (72, 99.5)),
    ('DA 3.0',  'a-wave amplitude',      170,  'GREEN', (119, 238)),
    ('DA 3.0',  'a-wave implicit time',  15.2, 'GREEN', (13.5, 16)),
    ('DA 3.0',  'b-wave amplitude',      280,  'GREEN', (191, 404)),
    ('DA 3.0',  'b-wave implicit time',  52,   'GREEN', (45.5, 58.5)),
    ('LA 3.0',  'a-wave amplitude',      8,    'RED',   (12, 43)),
    ('LA 3.0',  'a-wave implicit time',  15.5, 'GREEN', (13, 15.5)),
    ('LA 3.0',  'b-wave amplitude',      40,   'RED',   (73, 227)),
    ('LA 3.0',  'b-wave implicit time',  29,   'GREEN', (27, 31.5)),
    ('LA 30 Hz', 'flicker amplitude',    22,   'RED',   (50, 166)),
    ('LA 30 Hz', 'flicker implicit time', 31,  'RED',   (24, 28)),
    ('LA 3.0',  'PhNR amplitude',        14.2, 'Not zoned'),
]

**Zones.**

In [4]:
table, signal = zone_table(ROWS, age=30)
print(table.drop(columns='Match').to_string(index=False))
print('\nComposite Traffic Light:', signal)
check(table)

Protocol             Parameter  Value                            Limits Zone (code) Zone (book)
 DA 0.01      b-wave amplitude  215.0   138 to 297 (L_lo 130, U_hi 319)       GREEN       GREEN
 DA 0.01  b-wave implicit time   84.0  72 to 99.5 (L_lo 66.5, U_hi 106)       GREEN       GREEN
  DA 3.0      a-wave amplitude  170.0   119 to 238 (L_lo 109, U_hi 261)       GREEN       GREEN
  DA 3.0  a-wave implicit time   15.2     13.5 to 16 (L_lo 13, U_hi 17)       GREEN       GREEN
  DA 3.0      b-wave amplitude  280.0   191 to 404 (L_lo 178, U_hi 421)       GREEN       GREEN
  DA 3.0  b-wave implicit time   52.0 45.5 to 58.5 (L_lo 42.5, U_hi 60)       GREEN       GREEN
  LA 3.0      a-wave amplitude    8.0       12 to 43 (L_lo 11, U_hi 49)         RED         RED
  LA 3.0  a-wave implicit time   15.5   13 to 15.5 (L_lo 13, U_hi 15.5)       GREEN       GREEN
  LA 3.0      b-wave amplitude   40.0     73 to 227 (L_lo 54, U_hi 260)         RED         RED
  LA 3.0  b-wave implicit time   29.0   

**The diagnostic signature (§19.7.3).** Counting the zones by light adaptation shows the cone-selective pattern: every RED value is photopic, and every scotopic value is GREEN. It is the mirror image of the rod-predominant pattern of Case 3.

In [5]:
import pandas as pd
table['Adaptation'] = table['Protocol'].str[:2].map({'DA': 'Scotopic (DA)', 'LA': 'Photopic (LA)'})
zoned = table[table['Zone (code)'] != 'NOT ZONED']
print(pd.crosstab(zoned['Adaptation'], zoned['Zone (code)']))

Zone (code)    GREEN  RED
Adaptation               
Photopic (LA)      2    4
Scotopic (DA)      6    0


**Largest SHAP contributors (§19.7.3).** Constructed for teaching. The three largest are photopic; the normal DA 3.0 b-wave counts against RP, CSNB and DR, which affect rod-mediated responses.

In [6]:
for name, v in [('LA 30 Hz flicker amplitude', 0.29), ('LA 3.0 b-wave amplitude', 0.24),
                ('LA 30 Hz implicit time', 0.18), ('DA 3.0 b-wave amplitude (normal)', 0.11)]:
    print(f'{name:34s} {v:+.2f}')

LA 30 Hz flicker amplitude         +0.29
LA 3.0 b-wave amplitude            +0.24
LA 30 Hz implicit time             +0.18
DA 3.0 b-wave amplitude (normal)   +0.11
